# Team 11 Project    
# DreamerV3 Implementation & Extension   

**Course**: Deep Reinforcement Learning (AI.61100_2026_1)  
**Base Paper**: [*Mastering Diverse Domains through World Models (DreamerV3)*](https://arxiv.org/abs/2301.04104)  
**Reference Papers**: 
* [*World Models*](https://arxiv.org/abs/1803.10122)
* [*Learning Latent Dynamics for Planning from Pixels*](https://arxiv.org/abs/1811.04551)
* [*Mastering Atari with Discrete World Models*](https://arxiv.org/abs/2010.02193)

---

### Team 11 Members
* Hyeonseo Yun
* Kihyun Seol
* Seungmin Cha
* Seungyeon Ryu

### Reproducible local setup (GitHub/Hugging Face)

Run these steps in a **fresh local clone** (no `/mnt/...` assumptions):

1. Clone branch `dreamerv2-v3` and create Python 3.11 environment.
2. Install deps: `pip install -U -r requirements.txt huggingface_hub gymnasium highway-env imageio ruamel.yaml`.
3. (Optional) set HF token: `export HF_TOKEN=...` for private/quota-safe downloads.
4. Open this notebook at repo root and run cells top-to-bottom.
5. Default mode uses cached artifacts when present; set env flags only when needed:
   - `RUN_MC_LIVE=1` for live Minecraft rollout
   - `REFRESH_ATARI_GIFS=1` for 3-game Atari GIF refresh
   - `RUN_ATARI_LIVE=1` for timed retraining pipeline
6. Highway checkpoint is fetched from HF repo `HyunseoYun/dreamerv3-custom-envs` automatically when missing.

Expected outputs:
- `report/REPORT.md`, `report/ANALYSIS.md`
- `report/atari_compare/ATARI_COMPARE.md`
- `highlights/inference/*.gif` and `highway_roundabout_inference.gif`


In [1]:
import io, os, pathlib, sys, urllib.request, zipfile
from IPython.display import Image, display, Markdown
%matplotlib inline

REPO, BRANCH = 'franktome/Dreamerv3_RL_project', 'dreamerv2-v3'
WORKSPACE = pathlib.Path('.').resolve()
if pathlib.Path('dvbench/__init__.py').exists():
    sys.path.insert(0, str(WORKSPACE))
else:
    cache = pathlib.Path.home() / '.cache' / 'dvbench_pkg'
    cache.mkdir(parents=True, exist_ok=True)
    url = f'https://github.com/{REPO}/archive/refs/heads/{BRANCH}.zip'
    with urllib.request.urlopen(url, timeout=180) as r:
        data = r.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(cache)
    sys.path.insert(0, str(next(cache.glob(f'*-{BRANCH}'))))

from dvbench.paths import default_paths
from dvbench.env_setup import clone_dreamerv3, setup_jax, ensure_xvfb
from dvbench.hf_assets import login_if_needed, resolve_minecraft_logdir
from dvbench import inference_demo
from dvbench import viz_advanced

login_if_needed()
cfg = default_paths(WORKSPACE, gpu='1')
clone_dreamerv3(cfg)
setup_jax(cfg)
MC_LOGDIR = resolve_minecraft_logdir(cfg)
print('Minecraft logdir:', MC_LOGDIR)



DreamerV3 found: /mnt/server12_hard0/kiseol/Dreamerv3/vendor/dreamerv3
JAX devices: [CudaDevice(id=0)]
Minecraft logdir: /mnt/server12_hard0/kiseol/Dreamerv3/vendor/dreamerv3/logdir/minecraft_diamond_full


## 1.1 — Benchmark Results: Minecraft & Atari 57

Official published scores for Minecraft (V3 / PPO / IMPALA) and Atari 57 (V2 vs V3).
Local Minecraft training episodes overlaid when available.

### Metric: Human Normalized Score (HNS)

We use **HNS** to compare Atari agents on a common scale. For each game $g$, let $S$ be the agent score, $H_g$ the **human gamer** baseline, and $R_g$ the **random** baseline (from `baselines.json`):

$$
\mathrm{HNS}_g(S)=\frac{S-R_g}{H_g-R_g}
$$

- $\mathrm{HNS}_g=0$: random-level performance
- $\mathrm{HNS}_g=1$: human-level performance
- $\mathrm{HNS}_g>1$: above human

**How it appears in the plots below** (`dvbench/viz.py` (via `seollab`)):

1. **Per-game learning curves** — at each training budget $x$, the published score $S_x$ is converted via the formula above.
2. **Atari 57 median curve** — at each $x$, take the median over all games:
   $$
   \mathrm{MedianHNS}(x)=\mathrm{median}_{g}\,\mathrm{HNS}_g\!\left(S_{g,x}\right)
   $$
3. **Bar charts / win rate** — use $\mathrm{HNS}_g$ at the **50M-step** budget (last score with $x\le 50{,}000{,}000$).



In [ ]:
adv = viz_advanced.run_advanced_viz(cfg, logdir=MC_LOGDIR, show=False)
for key in ['minecraft_enhanced', 'minecraft_overlay', 'atari_v2_v3', 'atari_heatmap', 'atari_10_curves', 'atari_10_panels']:
    p = adv.get(key, '')
    if p and pathlib.Path(p).exists():
        display(Image(filename=p))
display(Markdown(
    f"**Atari median HNS:** V2={adv['median_hns_v2']:.2f} → V3={adv['median_hns_v3']:.2f} "
    f"({adv['v3_win_pct']:.0f}% games favor V3)"
))
display(Markdown(f"**10-game set:** {', '.join(adv['games'])}"))



/mnt/server12_hard0/kiseol/Dreamerv3/seollab/viz.py:133: RuntimeWarning: Mean of empty slice
  vals.append(np.nanmean(sc))
/mnt/server12_hard0/kiseol/Dreamerv3/seollab/viz.py:134: RuntimeWarning: All-NaN slice encountered
  meds.append(np.nanmedian(vals) if vals else np.nan)


Median HNS V2: 1.1114864864864864 V3: 4.037871168947268
V3 wins: 81.1%


### ✅ 1.1 Analysis

#### Minecraft — metric definitions (DreamerV3 paper & official scores)

**Milestone episode score** (MineRL `minecraft_diamond`, Hafner et al. 2023 / Nature 2025).

Each of 12 items grants a **one-time** sparse reward the first time it appears in inventory:

$$r_j = \mathbb{1}\big[\text{first\_collect}(m_j)\big], \quad j=1,\ldots,12$$

$$M = (\text{log},\text{planks},\text{stick},\text{crafting\_table},\text{wooden\_pickaxe},\text{cobblestone},\text{stone\_pickaxe},\text{iron\_ore},\text{furnace},\text{iron\_ingot},\text{iron\_pickaxe},\text{diamond})$$

Episode return (logged as `ys` in official JSON):

$$R = \sum_{j=1}^{12} r_j + \varepsilon_{\text{health}} \in [0,12]$$

where $\varepsilon_{\text{health}}$ is a small health term. **Diamond completion** requires $R \ge 12$ (paper Fig. 5: “discover diamond”).

---

**Left plot — Task success (%)**

For each training seed $s$, let $Y_s(t)$ be the **best episode score** seen up to environment step $t$:

$$\text{Success}_s(t)=\mathbb{1}\big[Y_s(t)\ge 12\big]$$

$$\text{TaskSuccess}(t)=\frac{100}{N}\sum_{s=1}^{N}\text{Success}_s(t)$$

This matches `danijar/dreamerv3` score plotting (`best >= 12`).

| Method | Seeds $N$ | $\text{TaskSuccess}(\infty)$ | Interpretation |
|--------|----------:|-----------------------------:|----------------|
| DreamerV3 | 20 | **70%** (14/20) | 14 seeds ever reach diamond |
| PPO | 14 | **0%** (0/14) | no seed reaches $R\ge12$ |
| IMPALA | 15 | **0%** (0/15) | no seed reaches $R\ge12$ |

<small>PPO/IMPALA at 0% on the left is **correct** on official JSON — not a plotting bug.</small>

---

**Right plot — Mean max milestone index**

$$\bar{M}(t)=\frac{1}{N}\sum_{s=1}^{N}\min\!\big(12,\;Y_s(t)\big)$$

This measures **average progress depth**, not binary completion. A seed stuck at iron pickaxe ($Y\approx 11$) contributes $\approx 11$ to the average.

| Method | Final $\bar{M}$ | Deepest item typically reached |
|--------|----------------:|--------------------------------|
| DreamerV3 | **11.95** | diamond on many seeds |
| PPO | **10.92** | mostly iron ingot / iron pickaxe (11/14 reach index $\ge 11$) |
| IMPALA | **11.00** | iron pickaxe on all 15 seeds, **never** diamond |

---

**Why left ≈ 0% but right looks similar?**

The two axes answer **different questions**:

| | Left: TaskSuccess | Right: $\bar{M}$ |
|--|-------------------|-------------------|
| Type | **binary** ($R\ge12$?) | **continuous** average depth |
| PPO/IMPALA | fail final milestone (diamond) | still reach depth $\sim$10–11 |
| Visual gap | large (0% vs 70%) | small (10.9–11.0 vs 12) |

Last milestone gap (paper): *“baselines progress up to iron pickaxe; **none discovers diamond**”* — only DreamerV3 crosses the $R\ge12$ threshold.

---

**Algorithmic difference (why depth $\approx 11$ but not 12?)**

| | DreamerV3 | PPO / IMPALA |
|--|-----------|--------------|
| Learning | **World model** (RSSM) + **imagination** in latent space | **Model-free** actor–critic |
| Horizon | plans multi-step crafts via predicted futures | strong reactive play, weaker on final sparse step |
| Diamond step | needs long exploration (caves, iron tool) after $\sim$20 min human-equivalent chain | baselines master early chain (high $\bar{M}$) but rarely complete step 12 |

---

- **Atari:** V3 improves median HNS and wins most per-game comparisons (see HNS note above).
- **Algorithms:** RSSM / reward heads / normalization differences are summarized at the **start of Part 2** (before B0 setup).
- **Minecraft (plots):** same official JSON as DreamerV3 repo; left = completion rate, right = mean depth — complementary, not contradictory.



## 1.2 Minecraft env rollout (DreamerV3)

Policy rollout from the local checkpoint. Milestone strip captures frames when inventory progress advances.

In [ ]:
from dvbench.inference_demo import (
    analyze_training_health,
    display_minecraft_inference,
    load_minecraft_rollout_index,
    run_minecraft_multi_rollouts,
)
from dvbench import inference_demo

RUN_LIVE = os.environ.get('RUN_MC_LIVE')
RUN_MULTI = os.environ.get('RUN_MC_MULTI')

if RUN_MULTI and (MC_LOGDIR / 'ckpt').exists():
    ensure_xvfb(cfg.display)
    cfg.apply_env(mem_fraction=0.42)
    mc_index = run_minecraft_multi_rollouts(
        cfg, logdir=MC_LOGDIR, n_rollouts=8, max_steps=3600, top_k=3)
elif RUN_LIVE and (MC_LOGDIR / 'ckpt').exists():
    ensure_xvfb(cfg.display)
    live = inference_demo.run_minecraft_env_gif(cfg, logdir=MC_LOGDIR, max_steps=3600)
    mc_index = {
        'ok': live.get('ok'),
        'best': live,
        'top_k': [{**live, 'rank': 1}],
        'checkpoint_id': live.get('checkpoint_id'),
        'training_step': live.get('training_step'),
        'n_rollouts': 1,
    }
else:
    mc_index = load_minecraft_rollout_index(cfg, logdir=MC_LOGDIR, top_k=3)

health = analyze_training_health(MC_LOGDIR)
mc_result = display_minecraft_inference(cfg, mc_index, health, logdir=MC_LOGDIR)


### ✅ 1.2 Analysis

- Episode **reward** reflects cumulative milestone index during the rollout.
- **Milestone strip** labels each inventory unlock; wider strip = more progress within the episode window.
- Compare with A1 official curves to see where this checkpoint sits relative to published V3 runs.

<small>

**Checkpoint & inference.** Rollout uses the **latest** checkpoint (`ckpt/latest`), not the best training episode. Training so far: **57 episodes / ~1.16M env steps** (A6000, ~20 policy fps). Best training run: **episode 18 @ 315k steps → furnace** (score 8.0); 39 later episodes did not beat it. A single inference rollout can show fewer events than that best episode.

**Why diamond is unrealistic here.** Official DreamerV3 reaches diamond at a **median ~52M steps** (furnace→diamond ~50M more). This run is still near **crafting_table / wooden_pickaxe** on most episodes and has never reached iron_ingot+. Even at ~20 fps, 52M steps ≈ **30 days** of continuous GPU time; the 100M-step budget ≈ **58 more days** from the current step count — with **no upward trend** toward deeper milestones, finishing diamond on one A6000 is not a practical project goal.

</small>


## 1.3 Atari Key Game Tasks Comparison

Ten games: eight largest V3 gains plus two where V2 remains competitive. Score trajectories from official 50M-step runs.

In [ ]:
display(Image(filename=str(cfg.report_dir / 'atari_10game_bars.png')))
top = adv['compare'].sort_values('Δ', ascending=False).head(5)[['DreamerV2', 'DreamerV3', 'Δ']]
top.index = [i.replace('atari_', '') for i in top.index]
top



### ✅ 1.3 Analysis

- **Full Atari 57 V2 vs V3** (median curve + all per-game bars) is shown once in **§1.1** — not repeated here.
- **10-game bars** below focus on the largest V3 gains plus two V2-competitive titles.
- **Δ** = DreamerV3 − DreamerV2 HNS @ 50M; positive Δ means V3 wins that game.
- Live env GIFs require per-game checkpoints; this notebook uses official score trajectories for reproducibility.

## 1.4 Executive summary

In [ ]:
from dvbench import report
display(Image(filename=str(cfg.report_dir / 'executive_summary.png')))
report.write_report(cfg, adv['mc_summary'], adv['compare'], local_mc=mc_result if mc_result.get('ok') else None, advanced=adv)
analysis = (cfg.report_dir / 'ANALYSIS.md').read_text(encoding='utf-8')
display(Markdown(analysis))
display(Markdown(f"Artifacts: `{cfg.report_dir / 'REPORT.md'}`"))



### ✅ 1.4 Analysis

- **Atari:** V3 improves median HNS and wins most per-game comparisons.
- **Minecraft:** V3 shows highest mean milestone depth among compared baselines on official data.
- **Local run:** supplements official curves with checkpoint-specific rollout behavior.

---

# 2. Atari Benchmark Evaluations: DreamerV2 vs DreamerV3 (3 Key Games)

Games: **pong**, **breakout**, **boxing** — inference (V2 left | V3 right) and environment perturbation evaluation.

| Mode | When to use |
|------|-------------|
| **Default (cached)** | Show existing GIFs under `highlights/inference/` |
| `REFRESH_ATARI_GIFS=1` | Re-run inference |
| `RUN_ATARI_LIVE=1` | Full 90 min/model retrain + infer |


### DreamerV2 vs DreamerV3 — Core Algorithm (brief)

Both agents are **model-based RL**: they learn a **world model (RSSM)**, imagine rollouts in latent space, and train an **actor–critic** on predicted rewards. Part 2 compares the same three Atari games under matched env-step budgets.

**References:** [DreamerV2](https://arxiv.org/abs/2010.02193) (Hafner et al., 2021) · [DreamerV3](https://arxiv.org/abs/2301.04104) (Hafner et al., 2023)

---

#### Shared training loop

1. **Encode** observation $x_t$ → tokens; **RSSM** maintains deterministic $h_t$ and stochastic $z_t$.
2. **Imagine** $H$ steps in latent space with the policy; predict rewards/continues.
3. **Actor–critic** optimizes on imagined $\lambda$-returns (both use $\lambda{=}0.95$).

**World-model objective (schematic, both versions):**

$$\mathcal{L}_\text{model} = \underbrace{-\ln p(x_t \mid h_t, z_t)}_{\text{reconstruction}} - \underbrace{\ln p(r_t \mid h_t, z_t)}_{\text{reward}} + \underbrace{\beta\,\mathrm{KL}\big[q(z_t|h_t,x_t)\,\|\,p(z_t|h_t)\big]}_{\text{latent regularization}}$$

---

#### Key differences (what changed from V2 → V3)

| Component | DreamerV2 | DreamerV3 |
|-----------|-----------|-----------|
| Observation scaling | task-tuned preprocessing | **symlog** on encoder inputs |
| Reward / value heads | **MSE (Gaussian)** | **symexp two-hot** (255 bins) |
| Return normalization | **StreamNorm** (EMA) during imagination | **percentile norm** (5th–95th, `retnorm`) |
| Episode continuation | optional discount head | explicit **continue head** (binary) |
| KL in RSSM | forward/reverse **balance** + free bits | **dyn** / **rep** split + **free nats** $\tau$ |
| Hyperparameters | tuned per benchmark suite | **fixed** across domains (paper claim) |

**Symlog / symexp** (V3 code: `embodied/jax/nets.py`):

$$\mathrm{symlog}(x)=\mathrm{sign}(x)\ln(1+|x|), \qquad \mathrm{symexp}(x)=\mathrm{sign}(x)\,(e^{|x|}-1)$$

**V3 KL terms** (separate dynamics vs representation; $\tau{=}$ `free_nats`, default 1.0):

$$\mathcal{L}_\text{dyn} = \max\!\big(\mathrm{KL}[q(z_t|h_t,x_t)\,\|\,p(z_t|h_t)],\;\tau\big), \quad \mathcal{L}_\text{rep} = \max\!\big(\mathrm{KL}[q\,\|\,p],\;\tau\big)$$

---

#### Actor–critic (shared $\lambda$-return, different normalization)

**$\lambda$-return** (both V2 and V3, $\lambda{=}0.95$):

$$R^\lambda_t = r_t + \gamma_t\bigl((1-\lambda)\,V_{t+1} + \lambda\,R^\lambda_{t+1}\bigr)$$

**DreamerV3 policy loss** (normalized advantage $\hat{A}_t = (R^\lambda_t - V_t)\,/\,\text{scale}$, entropy weight $\alpha$):

$$\mathcal{L}_\pi = -w_t\bigl(\log\pi(a_t|s_t)\cdot \hat{A}_t + \alpha\,\mathcal{H}(\pi)\bigr)$$

DreamerV2 uses the same imagination + $\lambda$-return backbone; the actor may backprop through imagined returns (**dynamics gradient**) or via $\log\pi \cdot A_t$ (**REINFORCE**), depending on action space.

**V3 takeaway:** symlog inputs + distributional reward/value + percentile return normalization make a **single hyperparameter set** work across Atari, Minecraft, and continuous control — the robustness gap visible in Part 1 HNS curves.

---

#### Link to this notebook

- **Part 1:** official 50M-step scores (V2 vs V3 median HNS and per-game $\Delta$).
- **Part 2 (below):** same games, matched env steps — checkpoint GIFs and §2.3 perturbations probe **learning speed** and **dynamics robustness** of the two world models.


In [ ]:
# B0 — setup
from pathlib import Path
from IPython.display import Image, display, Markdown
import os

from dvbench.paths import default_paths
from dvbench import atari_compare, atari_anim
from dvbench.atari_anim import AnimSpec, ANIM_PRESETS

cfg = default_paths(Path('.').resolve(), gpu='1')
cfg.apply_env(mem_fraction=0.25)
GAMES = atari_compare.GAMES
display(Markdown(f"**Games:** {', '.join(GAMES)}"))



## 2.1 — Training & inference pipeline

Runs smoke-tested timed training (V2 TF + V3 JAX sequential) then compare GIFs.

In [ ]:
RUN_LIVE = bool(os.environ.get('RUN_ATARI_LIVE'))
REFRESH = bool(os.environ.get('REFRESH_ATARI_GIFS'))

if RUN_LIVE:
    smoke = atari_compare.run_smoke(cfg)
    display(Markdown(f"Smoke: **{'OK' if smoke['ok'] else 'FAILED'}**"))
    if smoke['ok']:
        atari_compare.run_timed_training(cfg, minutes_per_run=90)
        pipeline = atari_compare.run_full_pipeline(cfg, smoke=False, skip_train=True)
elif REFRESH:
    display(Markdown('**Refreshing** compare + anim GIFs from current checkpoints (no retrain)…'))
    from dvbench import gif_compare, viz_atari_compare
    refresh = {'compare': {}, 'anim': {}}
    for game in GAMES:
        refresh['compare'][game] = gif_compare.infer_both(cfg, game, max_steps=1500)
        refresh['anim'][game] = {}
        for preset in ('fast', 'sluggish'):
            refresh['anim'][game][preset] = atari_anim.run_anim_compare(
                cfg, game, preset=preset, max_steps=1200)
    metrics = atari_compare.collect_metrics(cfg)
    viz_atari_compare.generate_all(cfg, metrics, GAMES)
    viz_atari_compare.write_atari_compare_report(cfg, metrics, refresh)
    pipeline = {'refreshed': True, 'games': list(GAMES)}
    display(Markdown('Refresh complete.'))
else:
    cache = cfg.report_dir / 'atari_compare' / 'pipeline_result.json'
    pipeline = {'cached': True, 'report': str(cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md')}
    if cache.exists():
        display(Markdown(f"Using cached pipeline metadata: `{cache}`"))
    else:
        display(Markdown(
            'Using cached GIFs/plots on disk. Set `REFRESH_ATARI_GIFS=1` after long training '
            'to regenerate side-by-side GIFs from latest checkpoints.'
        ))

display(Markdown(f"Report: `{cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md'}`"))



## 2.1b Fair Atari Retrain (100k env steps)

Controlled retrain of **pong / breakout / boxing** with identical settings (`atari atari_compare`, repeat=4, sticky=0.25, noops=30, grayscale). Budget: **100,000 environment steps** per run (~400k logged frames). Comparison uses `aligned_env_steps = min(V2, V3)`; GIFs load V2 sidecar snapshots and V3 checkpoints at or before the aligned step.

Checkpoints: [Hugging Face `checkpoints/atari_*`](https://huggingface.co/HyunseoYun/dreamerv3-custom-envs/tree/main/checkpoints)

In [ ]:
# Fair retrain summary (aligned checkpoint inference)
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

from dvbench import atari_align, atari_compare, gif_compare

align_df = pd.DataFrame(atari_align.alignment_table(cfg, GAMES))

display(Markdown('### Alignment (fair cutoff per game)'))
show = align_df[['game', 'v2_env_steps', 'v2_episodes', 'v3_env_steps', 'v3_episodes',
                 'aligned_env_steps', 'aligned_episodes', 'fair_gif']].copy()
show.columns = ['game', 'V2 steps', 'V2 ep', 'V3 steps', 'V3 ep', 'aligned steps', 'aligned ep', 'fair GIF']
display(show)

metrics = atari_compare.collect_metrics(cfg)
display(Markdown('### Metrics at aligned steps'))
display(metrics['summary'])

lc = cfg.report_dir / 'atari_compare' / 'learning_curves_3games.png'
if lc.exists():
    display(Markdown('### Learning curves (truncated to aligned steps)'))
    display(Image(filename=str(lc)))

display(Markdown('### Side-by-side GIFs (aligned checkpoints)'))
for game in GAMES:
    row = align_df[align_df['game'] == game].iloc[0]
    cmp = gif_compare.infer_both(cfg, game, max_steps=1500, align=True)
    v2s = cmp.get('v2', {}).get('score') if cmp.get('ok') else None
    v3s = cmp.get('v3', {}).get('score') if cmp.get('ok') else None
    score_txt = ''
    if v2s is not None and v3s is not None:
        score_txt = f' | rollout V2={v2s} V3={v3s}'
    status = '✅ fair GIF' if row['fair_gif'] and cmp.get('ok') else '⚠️ check alignment'
    display(Markdown(
        f"**{game}** — aligned **{int(row['aligned_env_steps']):,}** env steps "
        f"(V2 ep {int(row['v2_episodes'])}, V3 ep {int(row['v3_episodes'])}) "
        f"{status}{score_txt}"
    ))
    gif_path = cmp.get('compare_gif') or (cfg.highlights_dir / 'inference' / f'atari_{game}_v2v3_compare.gif')
    if cmp.get('ok') and Path(gif_path).exists():
        display(Image(filename=str(gif_path)))
    elif not cmp.get('ok'):
        display(Markdown(f"*{game}: aligned inference failed*"))

display(Markdown(f"Full report: `{cfg.report_dir / 'atari_compare' / 'ATARI_COMPARE.md'}`"))
display(Markdown(f"Alignment details: `{cfg.report_dir / 'atari_compare' / 'ALIGNMENT.md'}`"))


## 2.2 — Comparison Between DreamerV2 & DreamerV3

*Algorithm snapshot: see **DreamerV2 vs DreamerV3 — Core Algorithm** above (Part 2 intro).*


In [ ]:
from datetime import datetime
from pathlib import Path

from dvbench import atari_align, gif_compare

def _mtime(p):
    return datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M') if p.exists() else 'missing'

align_rows = {r['game']: r for r in atari_align.alignment_table(cfg, GAMES)}

for game in GAMES:
    al = align_rows[game]
    cmp = gif_compare.infer_both(cfg, game, max_steps=1500, align=True)
    v2s = cmp.get('v2', {}).get('score') if cmp.get('ok') else None
    v3s = cmp.get('v3', {}).get('score') if cmp.get('ok') else None
    fair = '✅ fair GIF' if al['fair_gif'] and cmp.get('ok') else '⚠️ check alignment'
    score_txt = f' | rollout V2={v2s} V3={v3s}' if v2s is not None and v3s is not None else ''
    gif_path = cmp.get('compare_gif') or (cfg.highlights_dir / 'inference' / f'atari_{game}_v2v3_compare.gif')
    if cmp.get('ok') and Path(gif_path).exists():
        display(Markdown(
            f"### {game} — V2|V3 @ **{al['aligned_env_steps']:,}** env steps {fair}\n"
            f"gif {_mtime(Path(gif_path))} | V2 env {al['v2_env_steps']:,} | V3 ckpt {al.get('v3_ckpt_env_steps', '—')}{score_txt}"
        ))
        display(Image(filename=str(gif_path)))
    else:
        display(Markdown(
            f"*{game}: aligned inference failed — check checkpoints at aligned step {al['aligned_env_steps']:,}*"
        ))


<small>

In the V2|V3 compare GIFs above, DreamerV2 rollouts tend to show **low action entropy**, so the policy rarely switches actions. **Frame counts may also differ** between V2 and V3 (e.g. breakout), which can desync the two sides for part of the clip.

</small>


## 2.3 Environment Perturbations Evaluation

Each GIF stacks **baseline V2|V3** (top) and **perturbed V2|V3** (bottom).

- **Cached presets:** `baseline`, `fast`, `sluggish`
- **Interactive explorer** below: switch game/preset, tune `repeat`/`sticky` sliders (snaps to nearest cached preset — **no live GIF regen**).


In [ ]:
from dvbench import atari_anim_ui

# Interactive UI: game + preset buttons + repeat/sticky sliders → cached GIF + score bars
atari_anim_ui.show_perturb_explorer(cfg, GAMES)

# Static fallback (if widgets unavailable)
for game in GAMES:
    for preset in ('fast', 'sluggish'):
        gif = cfg.highlights_dir / 'inference' / 'anim' / f'atari_{game}_{preset}_v2v3.gif'
        if gif.exists():
            display(Markdown(f"**{game} — {preset}** (cached)"))
            display(Image(filename=str(gif)))


### ✅ 2.3 Analysis

- **fast** (`repeat=2`): shorter frame skip → faster ball/paddle dynamics.
- **sluggish** (`sticky=0.5`): actions repeat more often → delayed response.
- Use the explorer to compare how **the same checkpoint** behaves under each cached perturbation.
- Sliders map to the nearest preset; re-run B1 with `REFRESH_ATARI_GIFS=1` to refresh GIFs and `anim_index.json` scores.


----

# 3. Highway Environment Inference & Evaluation

Beyond the **150 diverse tasks** evaluated in the main DreamerV3 paper, this section presents the inference and evaluation results after training on a novel task: the highway environment.

In [ ]:
# Render directly without pyvirtualdisplay
import gymnasium as gym
import highway_env

env = gym.make('roundabout-v0', render_mode='rgb_array')
print(env.observation_space.shape)
print(env.action_space.n)
obs, _ = env.reset()
frame = env.render()  # Returns a numpy array (headless)
print(frame.shape)    # (H, W, 3) indicates success


In [ ]:
# Cell 1 - no pyvirtualdisplay; env vars only
# JAX/XLA writes PTX to /tmp during GPU compile; if root disk is full,
# Agent init raises RESOURCE_EXHAUSTED. Run before importing jax.
import sys
import os
import importlib
import pathlib as _pl

WORKSPACE = _pl.Path('.').resolve()
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

# Reload if a prior cell cached an older dvbench/seollab
for _mod in ('dvbench.paths', 'dvbench', 'seollab.paths', 'seollab'):
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

from dvbench.paths import default_paths
cfg = default_paths(WORKSPACE, gpu='0')
cfg.apply_env()  # sets TMPDIR / JAX cache
print('TMPDIR:', os.environ['TMPDIR'])

import jax
print('JAX devices:', jax.devices())




In [ ]:
import elements

# Hardcoded config dict (replaces YAML loading)
config_dict = {
    "loss_scales": {
        "rec": 1.0, "rew": 1.0, "con": 1.0, "dyn": 1.0,
        "rep": 0.1, "policy": 1.0, "value": 1.0, "repval": 0.3
    },
    "opt": {
        "lr": 4e-05, "agc": 0.3, "eps": 1e-20, "beta1": 0.9,
        "beta2": 0.999, "momentum": True, "wd": 0.0,
        "schedule": "const", "warmup": 1000, "anneal": 0
    },
    "ac_grads": False,
    "dyn": {
        "typ": "rssm",
        "rssm": {
            "deter": 2048, "hidden": 256, "stoch": 32, "classes": 16,
            "act": "silu", "norm": "rms", "unimix": 0.01,
            "outscale": 1.0, "winit": "trunc_normal_in",
            "imglayers": 2, "obslayers": 1, "dynlayers": 1,
            "absolute": False, "blocks": 8, "free_nats": 1.0
        }
    },
    "enc": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "winit": "trunc_normal_in", "symlog": True,
            "outer": False, "kernel": 5, "strided": False
        }
    },
    "dec": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "outscale": 1.0, "winit": "trunc_normal_in",
            "outer": False, "kernel": 5, "bspace": 8, "strided": False
        }
    },
    "rewhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "conhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "binary", "outscale": 1.0, "winit": "trunc_normal_in"
    },
    "policy": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "minstd": 0.1, "maxstd": 1.0, "outscale": 0.01,
        "unimix": 0.01, "winit": "trunc_normal_in"
    },
    "value": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "policy_dist_disc": "categorical",
    "policy_dist_cont": "bounded_normal",
    "imag_last": 0,
    "imag_length": 15,
    "horizon": 333,
    "contdisc": True,
    "imag_loss": {"slowtar": False, "lam": 0.95, "actent": 0.0003, "slowreg": 1.0},
    "repl_loss": {"slowtar": False, "lam": 0.95, "slowreg": 1.0},
    "slowvalue": {"rate": 0.02, "every": 1},
    "retnorm": {"impl": "perc", "rate": 0.01, "limit": 1.0, "perclo": 5.0, "perchi": 95.0, "debias": False},
    "valnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "advnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "reward_grad": True,
    "repval_loss": True,
    "repval_grad": True,
    "report": True,
    "report_gradnorms": False,
    "logdir": "logdir/highway_roundabout",
    "seed": 0,
    "jax": {
        "platform": "cuda", "compute_dtype": "bfloat16",
        "policy_devices": [0], "train_devices": [0],
        "mock_devices": 0, "prealloc": True, "jit": True,
        "debug": False, "expect_devices": 0, "enable_policy": True,
        "coordinator_address": ""
    },
    "batch_size": 16,
    "batch_length": 64,
    "replay_context": 1,
    "report_length": 32,
    "replica": 0,
    "replicas": 1
}

config =elements.Config(
    config_dict
)


In [ ]:
# Cell 2 - load env and agent
import os
import sys
import importlib
import pathlib

for _m in ('dvbench.paths', 'seollab.paths'):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
from dvbench.paths import default_paths
default_paths('.').apply_env()
print('TMPDIR:', os.environ['TMPDIR'])

import gymnasium as gym
import highway_env
import numpy as np
import imageio
import elements
import ruamel.yaml as yaml
from dreamerv3.agent import Agent
from dvbench.hf_assets import ensure_checkpoints

DOWNLOAD_DIR = ensure_checkpoints(pathlib.Path('.'))

env = gym.make('roundabout-v0', render_mode='rgb_array')

obs_space = {
    "obs": elements.Space(np.float32, env.observation_space.shape),
    "reward": elements.Space(np.float32),
    "is_first": elements.Space(bool),
    "is_last": elements.Space(bool),
    "is_terminal": elements.Space(bool),
}
act_space = {"action": elements.Space(np.int32, (), 0, env.action_space.n)}

agent = Agent(obs_space, act_space, config)
cp = elements.Checkpoint(pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout')
cp.agent = agent
print('ckpt:', list((pathlib.Path(DOWNLOAD_DIR) / 'checkpoints/highway_roundabout').iterdir()))
cp.load()
print('Checkpoint loaded.')




In [ ]:
# Cell 3 - inference and save GIF
env = gym.make('roundabout-v0', render_mode='rgb_array', config={'duration': 100})
frames = []
obs_raw, _ = env.reset()
done = False
total_reward = 0
carry = agent.init_policy(1)
is_first = True

while not done:
    frame = env.render()
    frames.append(frame)

    obs = {
        "obs": np.array([obs_raw], dtype=np.float32),        # (1, 5, 5)
        "reward": np.array([0.0], dtype=np.float32),          # (1,)
        "is_first": np.array([is_first]),                     # (1,)
        "is_last": np.array([False]),                         # (1,)
        "is_terminal": np.array([False]),                     # (1,)
    }
    is_first = False

    carry, act, _ = agent.policy(carry, obs, mode='eval')
    action = int(act['action'][0])
    obs_raw, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    done = terminated or truncated

env.close()
print(f"Total reward: {total_reward:.2f}, frames: {len(frames)}")

imageio.mimsave('highway_roundabout_inference.gif', frames, fps=10)
print("GIF saved.")


In [ ]:
# Cell 4 - inline preview in notebook
from IPython.display import Image
Image('highway_roundabout_inference.gif')
